# Filter Pasien: Irisan Data Tabular ∩ Citra USG

**Tujuan.** Menyisakan hanya pasien yang *lengkap secara multimodal*, yaitu pasien yang memenuhi **kedua** syarat:

1. punya baris di `app_data.xlsx` **dan** `US_Number`-nya terisi (kunci penghubung ke citra), dan
2. punya minimal satu citra di `US_Pictures_clean/` (hasil `split_multiple_in_one.py`).

**Kunci mapping.** `US_Number` (tabular) = nomor pasien di depan nama file citra (`<subject>.<view> ...`), yang di manifest tersimpan pada kolom `US_Number`. Satu pasien boleh punya banyak citra (*bag* pada MIL), tetapi hanya punya satu baris tabular.

**Alur.**

| Langkah | Isi |
|---|---|
| 1 | Konfigurasi path |
| 2 | Muat data tabular, buang baris tanpa `US_Number` |
| 3 | Muat manifest citra, pastikan file benar-benar ada di disk |
| 4 | Hitung irisan pasien dan laporkan yang tersisih |
| 5 | Terapkan filter ke kedua tabel + validasi |
| 6 | Simpan hasil |

**Catatan.** File asli (`app_data.xlsx`, `US_Pictures/`, `US_Pictures_clean/images/`) **tidak diubah atau dihapus**. Filter hanya menghasilkan dua file CSV baru; model nantinya membaca dua CSV ini saja.

In [ ]:
# Cari root proyek agar notebook dapat dijalankan dari subfolder.
from pathlib import Path as _WorkspacePath
import os as _workspace_os
import sys as _workspace_sys
_workspace_root = next((p for p in [_WorkspacePath.cwd(), *_WorkspacePath.cwd().parents]
                        if (p / "multimodal_mil_model.py").is_file()
                        and (p / "app_data_filtered.csv").is_file()), None)
if _workspace_root is None:
    raise RuntimeError("Open this notebook from inside the project workspace.")
_workspace_os.chdir(_workspace_root)
if str(_workspace_root) not in _workspace_sys.path:
    _workspace_sys.path.insert(0, str(_workspace_root))

# ---------------------------------------------------------------------------
# Langkah 1 - Konfigurasi
# ---------------------------------------------------------------------------
from pathlib import Path

import pandas as pd

ROOT = Path(".")                                   # folder proyek (tempat notebook ini berada)
TABULAR_XLSX = ROOT / "app_data.xlsx"              # data tabular asli (sheet "All cases")
CLEAN_DIR = ROOT / "US_Pictures_clean"             # keluaran split_multiple_in_one.py
MANIFEST_CSV = CLEAN_DIR / "image_manifest.csv"    # satu baris = satu citra hasil akhir

# Keluaran notebook ini
PREPARATION_OUTPUT = ROOT
OUT_TABULAR = PREPARATION_OUTPUT / "app_data_filtered.csv"
OUT_MANIFEST = PREPARATION_OUTPUT / "image_manifest_filtered.csv"

KEY = "US_Number"  # kunci penghubung tabular <-> citra

In [ ]:
# ---------------------------------------------------------------------------
# Langkah 2 - Muat data tabular & buang baris tanpa US_Number
# ---------------------------------------------------------------------------
# US_Number di Excel bertipe float (882.0) dan ada yang kosong (NaN).
# Pasien tanpa US_Number tidak mungkin dipetakan ke citra -> dikeluarkan.
tab_raw = pd.read_excel(TABULAR_XLSX, sheet_name="All cases")
tab_raw[KEY] = pd.to_numeric(tab_raw[KEY], errors="coerce")

n_no_key = int(tab_raw[KEY].isna().sum())
tab = tab_raw.dropna(subset=[KEY]).copy()
tab[KEY] = tab[KEY].astype(int)  # float -> int agar bisa dicocokkan dengan manifest

# Sanity check: tiap pasien harus muncul sekali di tabular
assert tab[KEY].is_unique, "US_Number duplikat di tabular: join akan menggandakan baris"

print(f"Baris tabular awal          : {len(tab_raw)}")
print(f"  - tanpa US_Number (dibuang): {n_no_key}")
print(f"Baris tabular ber-US_Number : {len(tab)}")

Baris tabular awal          : 782
  - tanpa US_Number (dibuang): 22
Baris tabular ber-US_Number : 760


In [ ]:
# ---------------------------------------------------------------------------
# Langkah 3 - Muat manifest citra & verifikasi file di disk
# ---------------------------------------------------------------------------
# Manifest hanyalah daftar; "punya citra" harus berarti file PNG-nya benar-benar
# ada di US_Pictures_clean/. Baris manifest yang filenya hilang dibuang.
man_raw = pd.read_csv(MANIFEST_CSV)
assert man_raw["image_id"].is_unique, "image_id duplikat: ada file hasil split yang saling menimpa"

exists = man_raw["path"].map(lambda p: (CLEAN_DIR / p).is_file())
man = man_raw[exists].copy()
man[KEY] = man[KEY].astype(int)

print(f"Baris manifest              : {len(man_raw)}")
print(f"  - file tidak ada di disk   : {int((~exists).sum())}")
print(f"Citra valid                 : {len(man)}  (milik {man[KEY].nunique()} pasien)")

Baris manifest              : 2120
  - file tidak ada di disk   : 0
Citra valid                 : 2120  (milik 707 pasien)


In [ ]:
# ---------------------------------------------------------------------------
# Langkah 4 - Irisan pasien & laporan pasien yang tersisih
# ---------------------------------------------------------------------------
tab_ids = set(tab[KEY])   # pasien yang punya data tabular + US_Number
img_ids = set(man[KEY])   # pasien yang punya minimal satu citra valid

keep_ids = tab_ids & img_ids              # DIPERTAHANKAN: punya keduanya
only_tab = sorted(tab_ids - img_ids)      # ada di tabular, tidak punya citra
only_img = sorted(img_ids - tab_ids)      # punya citra, tidak ada di tabular

print(f"Pasien tabular (ber-US_Number) : {len(tab_ids)}")
print(f"Pasien dengan citra            : {len(img_ids)}")
print(f"-> DIPERTAHANKAN (irisan)      : {len(keep_ids)}")
print(f"-> dibuang, tabular tanpa citra: {len(only_tab)}")
print(f"-> dibuang, citra tanpa tabular: {len(only_img)} "
      f"({int(man[KEY].isin(only_img).sum())} citra)")

Pasien tabular (ber-US_Number) : 760
Pasien dengan citra            : 707
-> DIPERTAHANKAN (irisan)      : 579
-> dibuang, tabular tanpa citra: 181
-> dibuang, citra tanpa tabular: 128 (375 citra)


In [ ]:
# ---------------------------------------------------------------------------
# Langkah 5 - Terapkan filter ke kedua tabel + validasi
# ---------------------------------------------------------------------------
tab_f = tab[tab[KEY].isin(keep_ids)].sort_values(KEY).reset_index(drop=True)
man_f = man[man[KEY].isin(keep_ids)].sort_values([KEY, "view", "panel", "image_id"]).reset_index(drop=True)

# --- Validasi: kedua tabel harus memuat himpunan pasien yang SAMA persis ---
assert set(tab_f[KEY]) == set(man_f[KEY]) == keep_ids
assert tab_f[KEY].is_unique                      # 1 pasien = 1 baris tabular
assert man_f["image_id"].is_unique               # tiap citra unik
assert man_f["path"].map(lambda p: (CLEAN_DIR / p).is_file()).all()  # semua file ada
assert man_f.groupby(KEY).size().min() >= 1      # tiap pasien >= 1 citra

# --- Ringkasan: ukuran "bag" per pasien (relevan untuk MIL) ---
bag = man_f.groupby(KEY).size()
print(f"Tabular hasil filter : {tab_f.shape[0]} pasien x {tab_f.shape[1]} kolom")
print(f"Citra hasil filter   : {len(man_f)} citra")
print(f"Citra per pasien     : min={bag.min()}, median={bag.median():.0f}, "
      f"rata-rata={bag.mean():.1f}, maks={bag.max()}")
print("\nDistribusi target (pasien yang dipertahankan):")
for col in ["Diagnosis", "Management", "Severity"]:
    print(f"\n{col}\n{tab_f[col].value_counts(dropna=False).to_string()}")

Tabular hasil filter : 579 pasien x 58 kolom
Citra hasil filter   : 1745 citra
Citra per pasien     : min=1, median=2, rata-rata=3.0, maks=15

Distribusi target (pasien yang dipertahankan):

Diagnosis
Diagnosis
appendicitis       382
no appendicitis    196
NaN                  1

Management
Management
conservative          346
primary surgical      216
secondary surgical     17

Severity
Severity
uncomplicated    483
complicated       96


In [ ]:
# ---------------------------------------------------------------------------
# Langkah 6 - Simpan hasil
# ---------------------------------------------------------------------------
# Disimpan sebagai CSV baru; app_data.xlsx dan folder citra tidak disentuh.
# Kedua file dihubungkan oleh kolom US_Number (relasi 1 pasien : N citra).
tab_f.to_csv(OUT_TABULAR, index=False)
man_f.to_csv(OUT_MANIFEST, index=False)

print(f"Tersimpan: {OUT_TABULAR}  ({len(tab_f)} baris)")
print(f"Tersimpan: {OUT_MANIFEST}  ({len(man_f)} baris)")

# Contoh pemakaian: ambil semua citra + label seorang pasien
contoh = int(tab_f[KEY].iloc[0])
print(f"\nContoh pasien US_Number={contoh}")
print(tab_f.loc[tab_f[KEY] == contoh, ["Age", "Sex", "Diagnosis", "Management", "Severity"]].to_string(index=False))
man_f.loc[man_f[KEY] == contoh, ["image_id", "label", "path"]]

Tersimpan: app_data_filtered.csv  (579 baris)
Tersimpan: image_manifest_filtered.csv  (1745 baris)

Contoh pasien US_Number=1
      Age  Sex       Diagnosis   Management      Severity
12.531143 male no appendicitis conservative uncomplicated


,image_id,label,path
0,S0001_V01_P01,RLQ_Pathological Lymphnodes,images/S0001_V01_P01.png
1,S0001_V02_P01,Ileitis,images/S0001_V02_P01.png
2,S0001_V03_P01,Mesenterial Lymphadenitis,images/S0001_V03_P01.png
3,S0001_V04_P01,Appendix,images/S0001_V04_P01.png
4,S0001_V04_P01_D02,Appendix,images/S0001_V04_P01_D02.png
5,S0001_V05_P01,"Appendix, surrounding tissue reaction",images/S0001_V05_P01.png
